<a href="https://colab.research.google.com/github/PrivateerDev/ColabBD/blob/main/Pr%C3%A1ctica1BD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

:

In [28]:
!pip -q install gradio pandas

In [29]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cur = conn.cursor()

cur.execute('''
CREATE TABLE productos_maiz (
    id INTEGER PRIMARY KEY,
    nombre TEXT,
    categoria TEXT,
    precio REAL,
    stock INTEGER
)
''')

cur.executemany('INSERT INTO productos_maiz VALUES (?,?,?,?,?)', [
    (1, 'Tortillas de maíz', 'Tortillería', 18.0, 100),
    (2, 'Esquites', 'Antojitos', 25.0, 40),
    (3, 'Elote en vaso', 'Antojitos', 30.0, 35),
])
conn.commit()
print('✅ Tabla productos_maiz lista')
pd.read_sql_query('SELECT * FROM productos_maiz', conn)

✅ Tabla productos_maiz lista


,id,nombre,categoria,precio,stock
0,1,Tortillas de maíz,Tortillería,18.0,100
1,2,Esquites,Antojitos,25.0,40
2,3,Elote en vaso,Antojitos,30.0,35


In [30]:
cur.executemany('INSERT INTO productos_maiz VALUES (?,?,?,?,?)', [
    (4, 'Tamales de rajas', 'Tamales', 20.0, 50),
    (5, 'Atole de guayaba', 'Bebidas', 22.0, 30),
    (6, 'Tostadas de maíz', 'Tortillería', 12.0, 80),
])
conn.commit()
print('✅ 3 productos insertados')
pd.read_sql_query('SELECT * FROM productos_maiz', conn)

✅ 3 productos insertados


,id,nombre,categoria,precio,stock
0,1,Tortillas de maíz,Tortillería,18.0,100
1,2,Esquites,Antojitos,25.0,40
2,3,Elote en vaso,Antojitos,30.0,35
3,4,Tamales de rajas,Tamales,20.0,50
4,5,Atole de guayaba,Bebidas,22.0,30
5,6,Tostadas de maíz,Tortillería,12.0,80


In [31]:
pd.read_sql_query('SELECT * FROM productos_maiz WHERE stock > 40', conn)

,id,nombre,categoria,precio,stock
0,1,Tortillas de maíz,Tortillería,18.0,100
1,4,Tamales de rajas,Tamales,20.0,50
2,6,Tostadas de maíz,Tortillería,12.0,80


In [32]:
cur.execute('UPDATE productos_maiz SET precio = 28.0 WHERE id = 2')
conn.commit()
print('✅ Precio de Esquites actualizado')
pd.read_sql_query('SELECT * FROM productos_maiz WHERE id = 2', conn)

✅ Precio de Esquites actualizado


,id,nombre,categoria,precio,stock
0,2,Esquites,Antojitos,28.0,40


In [33]:
cur.execute('DELETE FROM productos_maiz WHERE id = 6')
conn.commit()
print('✅ Tostadas de maíz eliminadas')

✅ Tostadas de maíz eliminadas


In [34]:
pd.read_sql_query('SELECT * FROM productos_maiz', conn)

,id,nombre,categoria,precio,stock
0,1,Tortillas de maíz,Tortillería,18.0,100
1,2,Esquites,Antojitos,28.0,40
2,3,Elote en vaso,Antojitos,30.0,35
3,4,Tamales de rajas,Tamales,20.0,50
4,5,Atole de guayaba,Bebidas,22.0,30


In [35]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect('ventas_maiz.db', check_same_thread=False)
cursor = conexion.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS ventas_maiz (
    id_producto INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    categoria TEXT NOT NULL,
    descripcion TEXT,
    precio REAL DEFAULT 0.0,
    stock INTEGER DEFAULT 0
)
''')

cursor.executemany(
    'INSERT OR IGNORE INTO ventas_maiz (nombre, categoria, descripcion, precio, stock) VALUES (?,?,?,?,?)',
    [
        ('Tortillas de maíz', 'Tortillería', 'Paquete de 1kg tortillas hechas a mano', 18.0, 100),
        ('Esquites', 'Antojitos', 'Vaso de esquites con chile, limón y mayonesa', 25.0, 40),
        ('Elote en vaso', 'Antojitos', 'Elote desgranado con crema y queso', 30.0, 35),
        ('Tamal de rajas', 'Tamales', 'Tamal de masa de maíz con rajas y queso', 20.0, 50),
        ('Atole de guayaba', 'Bebidas', 'Bebida caliente de masa de maíz con guayaba', 22.0, 30),
    ]
)
conexion.commit()
print('✅ Tabla ventas_maiz lista con 5 productos')
pd.read_sql_query('SELECT * FROM ventas_maiz', conexion)

✅ Tabla ventas_maiz lista con 5 productos


,id_producto,nombre,categoria,descripcion,precio,stock
0,1,Tortillas de maíz,Tortillería,Paquete de 1kg tortillas hechas a mano,18.0,100
1,2,Esquites,Antojitos,"Vaso de esquites con chile, limón y mayonesa",25.0,40
2,3,Elote en vaso,Antojitos,Elote desgranado con crema y queso,30.0,35
3,4,Tamal de rajas,Tamales,Tamal de masa de maíz con rajas y queso,20.0,50
4,5,Atole de guayaba,Bebidas,Bebida caliente de masa de maíz con guayaba,22.0,30


In [36]:
def mostrar_productos():
    return pd.read_sql_query('SELECT * FROM ventas_maiz ORDER BY id_producto', conexion)

def alta_producto(nombre, categoria, descripcion, precio, stock):
    if not nombre or not categoria:
        return 'Error: nombre y categoría son obligatorios.', mostrar_productos()
    try:
        cursor.execute(
            'INSERT INTO ventas_maiz (nombre, categoria, descripcion, precio, stock) VALUES (?,?,?,?,?)',
            (nombre, categoria, descripcion, float(precio), int(stock))
        )
        conexion.commit()
        return f'✅ Producto "{nombre}" registrado correctamente.', mostrar_productos()
    except Exception as e:
        return f'Error: {e}', mostrar_productos()

def baja_producto(id_producto):
    if not id_producto:
        return 'Error: ingresa un ID.', mostrar_productos()
    cursor.execute('DELETE FROM ventas_maiz WHERE id_producto = ?', (int(id_producto),))
    conexion.commit()
    if cursor.rowcount == 0:
        return f'No se encontró el producto con ID {id_producto}.', mostrar_productos()
    return f'✅ Producto ID {id_producto} eliminado.', mostrar_productos()

def cambio_producto(id_producto, nuevo_nombre, nueva_categoria, nueva_descripcion, nuevo_precio, nuevo_stock):
    if not id_producto:
        return 'Error: ingresa un ID.', mostrar_productos()
    cursor.execute(
        'UPDATE ventas_maiz SET nombre=?, categoria=?, descripcion=?, precio=?, stock=? WHERE id_producto=?',
        (nuevo_nombre, nueva_categoria, nueva_descripcion, float(nuevo_precio), int(nuevo_stock), int(id_producto))
    )
    conexion.commit()
    if cursor.rowcount == 0:
        return f'No se encontró el producto con ID {id_producto}.', mostrar_productos()
    return f'✅ Producto ID {id_producto} actualizado.', mostrar_productos()

def buscar_producto(texto):
    if not texto:
        return mostrar_productos()
    patron = f'%{texto}%'
    return pd.read_sql_query(
        'SELECT * FROM ventas_maiz WHERE nombre LIKE ? OR categoria LIKE ? OR descripcion LIKE ? ORDER BY id_producto',
        conexion, params=(patron, patron, patron)
    )

print('✅ Funciones CRUD listas')

✅ Funciones CRUD listas


In [37]:
import gradio as gr

with gr.Blocks(title='Tienda de Productos de Maíz') as demo:
    gr.Markdown('# 🌽 Sistema de Gestión — Productos de Maíz')
    gr.Markdown('### Tortillas • Esquites • Tamales • Antojitos')

    with gr.Tab('Altas'):
        gr.Markdown('## Registrar nuevo producto')
        nombre_a    = gr.Textbox(label='Nombre del producto')
        categoria_a = gr.Textbox(label='Categoría (ej. Antojitos, Tamales, Bebidas)')
        desc_a      = gr.Textbox(label='Descripción')
        precio_a    = gr.Number(label='Precio ($)', value=0)
        stock_a     = gr.Number(label='Stock (unidades)', value=0, precision=0)
        btn_alta    = gr.Button('Registrar producto')
        sal_alta    = gr.Textbox(label='Resultado')
        tbl_alta    = gr.Dataframe(label='Tabla actual', interactive=False)
        btn_alta.click(fn=alta_producto,
                       inputs=[nombre_a, categoria_a, desc_a, precio_a, stock_a],
                       outputs=[sal_alta, tbl_alta])

    with gr.Tab('Bajas'):
        gr.Markdown('## Eliminar un producto')
        id_baja  = gr.Number(label='ID del producto a eliminar', precision=0)
        btn_baja = gr.Button('Eliminar producto')
        sal_baja = gr.Textbox(label='Resultado')
        tbl_baja = gr.Dataframe(label='Tabla actual', interactive=False)
        btn_baja.click(fn=baja_producto, inputs=[id_baja], outputs=[sal_baja, tbl_baja])

    with gr.Tab('Cambios'):
        gr.Markdown('## Actualizar datos de un producto')
        id_c       = gr.Number(label='ID del producto', precision=0)
        nombre_c   = gr.Textbox(label='Nuevo nombre')
        cat_c      = gr.Textbox(label='Nueva categoría')
        desc_c     = gr.Textbox(label='Nueva descripción')
        precio_c   = gr.Number(label='Nuevo precio ($)', value=0)
        stock_c    = gr.Number(label='Nuevo stock', value=0, precision=0)
        btn_cambio = gr.Button('Actualizar producto')
        sal_cambio = gr.Textbox(label='Resultado')
        tbl_cambio = gr.Dataframe(label='Tabla actual', interactive=False)
        btn_cambio.click(fn=cambio_producto,
                         inputs=[id_c, nombre_c, cat_c, desc_c, precio_c, stock_c],
                         outputs=[sal_cambio, tbl_cambio])

    with gr.Tab('Búsquedas'):
        gr.Markdown('## Buscar productos')
        texto_b  = gr.Textbox(label='Buscar por nombre, categoría o descripción')
        btn_b    = gr.Button('Buscar')
        tbl_b    = gr.Dataframe(label='Resultados', interactive=False)
        btn_todos  = gr.Button('Mostrar todos')
        tbl_todos  = gr.Dataframe(label='Catálogo completo', interactive=False)
        btn_b.click(fn=buscar_producto, inputs=[texto_b], outputs=[tbl_b])
        btn_todos.click(fn=mostrar_productos, inputs=[], outputs=[tbl_todos])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5b13a111ccede7d443.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:

import sqlite3
import pandas as pd

conexion = sqlite3.connect("ejemplo_bd.db")
cursor = conexion.cursor()

print("Conexión realizada correctamente.")


Conexión realizada correctamente.


In [2]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS usuarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    correo TEXT NOT NULL,
    edad INTEGER
)
''')

conexion.commit()
print("Tabla 'usuarios' creada correctamente.")

Tabla 'usuarios' creada correctamente.


In [3]:

usuarios_iniciales = [
    ("Ana", "ana@gmail.com", 22),
    ("Luis", "luis@gmail.com", 25),
    ("María", "maria@gmail.com", 20),
    ("Carlos", "carlos@gmail.com", 28)
]

cursor.executemany(
    "INSERT INTO usuarios (nombre, correo, edad) VALUES (?, ?, ?)",
    usuarios_iniciales
)

conexion.commit()
print("Datos insertados correctamente.")

Datos insertados correctamente.


In [4]:
consulta = "SELECT * FROM usuarios"
df_usuarios = pd.read_sql_query(consulta, conexion)
df_usuarios

,id,nombre,correo,edad
0,1,Ana,ana@gmail.com,22
1,2,Luis,luis@gmail.com,25
2,3,María,maria@gmail.com,20
3,4,Carlos,carlos@gmail.com,28


In [5]:
consulta = "SELECT * FROM usuarios WHERE edad > 22"
df_filtrado = pd.read_sql_query(consulta, conexion)
df_filtrado

,id,nombre,correo,edad
0,2,Luis,luis@gmail.com,25
1,4,Carlos,carlos@gmail.com,28


In [6]:

cursor.execute('''
UPDATE usuarios
SET edad = 23
WHERE nombre = 'Ana'
''')

conexion.commit()
print("Registro actualizado correctamente.")

Registro actualizado correctamente.


In [7]:
pd.read_sql_query("SELECT * FROM usuarios", conexion)

,id,nombre,correo,edad
0,1,Ana,ana@gmail.com,23
1,2,Luis,luis@gmail.com,25
2,3,María,maria@gmail.com,20
3,4,Carlos,carlos@gmail.com,28


In [8]:
cursor.execute('''
DELETE FROM usuarios
WHERE nombre = 'Carlos'
''')

conexion.commit()
print("Registro eliminado correctamente.")



Registro eliminado correctamente.


In [9]:
pd.read_sql_query("SELECT * FROM usuarios", conexion)

,id,nombre,correo,edad
0,1,Ana,ana@gmail.com,23
1,2,Luis,luis@gmail.com,25
2,3,María,maria@gmail.com,20


In [10]:

cursor.execute('''
CREATE TABLE IF NOT EXISTS productos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    categoria TEXT NOT NULL,
    precio REAL NOT NULL,
    stock INTEGER NOT NULL
)
''')

conexion.commit()
print("Tabla 'productos' creada correctamente.")

Tabla 'productos' creada correctamente.


In [11]:

productos = [
    ("Laptop", "Tecnología", 14500.50, 10),
    ("Mouse", "Tecnología", 250.00, 35),
    ("Teclado", "Tecnología", 480.00, 20),
    ("Silla", "Mobiliario", 1800.00, 8)
]

cursor.executemany(
    "INSERT INTO productos (nombre, categoria, precio, stock) VALUES (?, ?, ?, ?)",
    productos
)

conexion.commit()
pd.read_sql_query("SELECT * FROM productos", conexion)

,id,nombre,categoria,precio,stock
0,1,Laptop,Tecnología,14500.5,10
1,2,Mouse,Tecnología,250.0,35
2,3,Teclado,Tecnología,480.0,20
3,4,Silla,Mobiliario,1800.0,8


In [12]:

pd.read_sql_query(
    "SELECT * FROM productos WHERE categoria = 'Tecnología'",
    conexion
)

,id,nombre,categoria,precio,stock
0,1,Laptop,Tecnología,14500.5,10
1,2,Mouse,Tecnología,250.0,35
2,3,Teclado,Tecnología,480.0,20


In [13]:
cursor.execute('''
UPDATE productos
SET stock = 15
WHERE nombre = 'Laptop'
''')
conexion.commit()

pd.read_sql_query("SELECT * FROM productos", conexion)

,id,nombre,categoria,precio,stock
0,1,Laptop,Tecnología,14500.5,15
1,2,Mouse,Tecnología,250.0,35
2,3,Teclado,Tecnología,480.0,20
3,4,Silla,Mobiliario,1800.0,8


In [14]:
cursor.execute('''
DELETE FROM productos
WHERE nombre = 'Mouse'
''')
conexion.commit()

pd.read_sql_query("SELECT * FROM productos", conexion)

,id,nombre,categoria,precio,stock
0,1,Laptop,Tecnología,14500.5,15
1,3,Teclado,Tecnología,480.0,20
2,4,Silla,Mobiliario,1800.0,8


## Parte A – Comprensión

1. **¿Qué es una base de datos?**
   Es un conjunto organizado de información almacenada de forma estructurada,
   que permite guardar, consultar, modificar y eliminar datos fácilmente.

2. **Tabla, campo y registro:**
   - **Tabla**: estructura que agrupa datos del mismo tipo (ej. "productos").
   - **Campo**: cada columna de l## Parte A – Comprensión

1. **¿Qué es una base de datos?**
   Es un conjunto organizado de información almacenada de forma estructurada,
   que permite guardar, consultar, modificar y eliminar datos fácilmente.

2. **Tabla, campo y registro:**
   - **Tabla**: estructura que agrupa datos del mismo tipo (ej. "productos").
   - **Campo**: cada columna de la tabla (ej. "precio", "stock").
   - **Registro**: cada fila con datos reales (ej. el producto Laptop).

3. **¿Para qué sirve una clave primaria?**
   Identifica de forma única a cada registro. No puede repetirse ni estar vacía.

4. **¿Qué hace SELECT?**
   Consulta y muestra datos de la tabla según los filtros que indiques.

5. **¿Qué hace UPDATE?**
   Modifica el valor de uno o más campos en registros ya existentes.a tabla (ej. "precio", "stock").
   - **Registro**: cada fila con datos reales (ej. el producto Laptop).

3. **¿Para qué sirve una clave primaria?**
   Identifica de forma única a cada registro. No puede repetirse ni estar vacía.

4. **¿Qué hace SELECT?**
   Consulta y muestra datos de la tabla según los filtros que indiques.

5. **¿Qué hace UPDATE?**
   Modifica el valor de uno o más campos en registros ya existentes.

In [15]:
cursor.executemany("INSERT INTO productos VALUES (?, ?, ?, ?, ?)", [
    (5,  'Monitor',   'Tecnología', 3200.0, 10),
    (6,  'Escritorio','Mobiliario', 4500.0,  3),
    (7,  'Mouse',     'Tecnología',  250.0, 30),
])
conexion.commit()
print("✅ 3 productos insertados")
pd.read_sql_query("SELECT * FROM productos", conexion)

✅ 3 productos insertados


,id,nombre,categoria,precio,stock
0,1,Laptop,Tecnología,14500.5,15
1,3,Teclado,Tecnología,480.0,20
2,4,Silla,Mobiliario,1800.0,8
3,5,Monitor,Tecnología,3200.0,10
4,6,Escritorio,Mobiliario,4500.0,3
5,7,Mouse,Tecnología,250.0,30


In [16]:
pd.read_sql_query("SELECT * FROM productos WHERE stock > 5", conexion)

,id,nombre,categoria,precio,stock
0,1,Laptop,Tecnología,14500.5,15
1,3,Teclado,Tecnología,480.0,20
2,4,Silla,Mobiliario,1800.0,8
3,5,Monitor,Tecnología,3200.0,10
4,7,Mouse,Tecnología,250.0,30


In [17]:
cursor.execute("UPDATE productos SET precio = 420.0 WHERE id = 3")
conexion.commit()
print("✅ Precio actualizado")
pd.read_sql_query("SELECT * FROM productos WHERE id = 3", conexion)

✅ Precio actualizado


,id,nombre,categoria,precio,stock
0,3,Teclado,Tecnología,420.0,20


In [18]:
cursor.execute("DELETE FROM productos WHERE id = 6")
conexion.commit()
print("✅ Producto eliminado")

✅ Producto eliminado


In [19]:
pd.read_sql_query("SELECT * FROM productos", conexion)

,id,nombre,categoria,precio,stock
0,1,Laptop,Tecnología,14500.5,15
1,3,Teclado,Tecnología,420.0,20
2,4,Silla,Mobiliario,1800.0,8
3,5,Monitor,Tecnología,3200.0,10
4,7,Mouse,Tecnología,250.0,30


In [20]:
pd.read_sql_query("SELECT * FROM productos", conexion)

,id,nombre,categoria,precio,stock
0,1,Laptop,Tecnología,14500.5,15
1,3,Teclado,Tecnología,420.0,20
2,4,Silla,Mobiliario,1800.0,8
3,5,Monitor,Tecnología,3200.0,10
4,7,Mouse,Tecnología,250.0,30


## Parte C – Análisis

Estas operaciones corresponden exactamente a lo que hace un sistema web real:

| Operación SQL | Acción en el sistema web              |
|---------------|---------------------------------------|
| INSERT        | Formulario de "Agregar producto"      |
| SELECT        | Pantalla de catálogo / búsqueda       |
| UPDATE        | Formulario de "Editar producto"       |
| DELETE        | Botón de "Eliminar" con confirmación  |

Cuando el usuario llena un formulario y da clic en "Guardar",
el backend ejecuta un INSERT o UPDATE en la base de datos.
Cuando se carga la lista de productos, se ejecuta un SELECT.

In [22]:
import sqlite3, pandas as pd

conn = sqlite3.connect(":memory:")
cur  = conn.cursor()

# Tabla principal con 5+ campos
cur.execute("""
CREATE TABLE clientes (
    id_cliente  INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre      TEXT    NOT NULL,
    apellido    TEXT    NOT NULL,
    email       TEXT    UNIQUE,
    telefono    TEXT,
    saldo       REAL    DEFAULT 0.0,
    activo      INTEGER DEFAULT 1
)
""")
conn.commit()
print("✅ Tabla 'clientes' creada")

✅ Tabla 'clientes' creada


In [23]:
cur.executemany("""
INSERT INTO clientes (nombre, apellido, email, telefono, saldo)
VALUES (?, ?, ?, ?, ?)
""", [
    ('Ana',    'García',   'ana@mail.com',    '5551001',  1500.00),
    ('Luis',   'Martínez', 'luis@mail.com',   '5551002',   800.50),
    ('María',  'López',    'maria@mail.com',  '5551003',  3200.00),
    ('Carlos', 'Ruiz',     'carlos@mail.com', '5551004',     0.00),
    ('Sofía',  'Torres',   'sofia@mail.com',  '5551005',  2100.75),
])
conn.commit()
pd.read_sql_query("SELECT * FROM clientes", conn)

,id_cliente,nombre,apellido,email,telefono,saldo,activo
0,1,Ana,García,ana@mail.com,5551001,1500.00,1
1,2,Luis,Martínez,luis@mail.com,5551002,800.50,1
2,3,María,López,maria@mail.com,5551003,3200.00,1
3,4,Carlos,Ruiz,carlos@mail.com,5551004,0.00,1
4,5,Sofía,Torres,sofia@mail.com,5551005,2100.75,1


In [24]:
cur.executemany("""
INSERT INTO clientes (nombre, apellido, email, telefono, saldo)
VALUES (?, ?, ?, ?, ?)
""", [
    ('Pedro', 'Sánchez', 'pedro@mail.com', '5551006', 500.0),
    ('Laura', 'Díaz',    'laura@mail.com', '5551007', 750.0),
])
conn.commit()
print("✅ 2 clientes dados de alta")
pd.read_sql_query("SELECT * FROM clientes", conn)

✅ 2 clientes dados de alta


,id_cliente,nombre,apellido,email,telefono,saldo,activo
0,1,Ana,García,ana@mail.com,5551001,1500.00,1
1,2,Luis,Martínez,luis@mail.com,5551002,800.50,1
2,3,María,López,maria@mail.com,5551003,3200.00,1
3,4,Carlos,Ruiz,carlos@mail.com,5551004,0.00,1
4,5,Sofía,Torres,sofia@mail.com,5551005,2100.75,1
5,6,Pedro,Sánchez,pedro@mail.com,5551006,500.00,1
6,7,Laura,Díaz,laura@mail.com,5551007,750.00,1


In [25]:
cur.execute("DELETE FROM clientes WHERE id_cliente = 4")
conn.commit()
print("✅ Cliente 4 eliminado")
pd.read_sql_query("SELECT * FROM clientes", conn)

✅ Cliente 4 eliminado


,id_cliente,nombre,apellido,email,telefono,saldo,activo
0,1,Ana,García,ana@mail.com,5551001,1500.00,1
1,2,Luis,Martínez,luis@mail.com,5551002,800.50,1
2,3,María,López,maria@mail.com,5551003,3200.00,1
3,5,Sofía,Torres,sofia@mail.com,5551005,2100.75,1
4,6,Pedro,Sánchez,pedro@mail.com,5551006,500.00,1
5,7,Laura,Díaz,laura@mail.com,5551007,750.00,1


In [26]:
# Cambio 1: actualizar saldo
cur.execute("UPDATE clientes SET saldo = 4000.0 WHERE id_cliente = 3")
# Cambio 2: actualizar teléfono
cur.execute("UPDATE clientes SET telefono = '5559999' WHERE id_cliente = 1")
conn.commit()
print("✅ 2 registros actualizados")
pd.read_sql_query("SELECT * FROM clientes", conn)

✅ 2 registros actualizados


,id_cliente,nombre,apellido,email,telefono,saldo,activo
0,1,Ana,García,ana@mail.com,5559999,1500.00,1
1,2,Luis,Martínez,luis@mail.com,5551002,800.50,1
2,3,María,López,maria@mail.com,5551003,4000.00,1
3,5,Sofía,Torres,sofia@mail.com,5551005,2100.75,1
4,6,Pedro,Sánchez,pedro@mail.com,5551006,500.00,1
5,7,Laura,Díaz,laura@mail.com,5551007,750.00,1


In [27]:
# Búsqueda 1: clientes con saldo mayor a 1000
print("── Clientes con saldo > $1,000 ──")
display(pd.read_sql_query("SELECT * FROM clientes WHERE saldo > 1000", conn))

# Búsqueda 2: buscar por nombre
print("── Buscar 'Ana' ──")
display(pd.read_sql_query("SELECT * FROM clientes WHERE nombre = 'Ana'", conn))

# Búsqueda 3: ordenar por saldo descendente
print("── Ranking por saldo ──")
display(pd.read_sql_query("SELECT nombre, apellido, saldo FROM clientes ORDER BY saldo DESC", conn))

── Clientes con saldo > $1,000 ──


,id_cliente,nombre,apellido,email,telefono,saldo,activo
0,1,Ana,García,ana@mail.com,5559999,1500.00,1
1,3,María,López,maria@mail.com,5551003,4000.00,1
2,5,Sofía,Torres,sofia@mail.com,5551005,2100.75,1


── Buscar 'Ana' ──


,id_cliente,nombre,apellido,email,telefono,saldo,activo
0,1,Ana,García,ana@mail.com,5559999,1500.0,1


── Ranking por saldo ──


,nombre,apellido,saldo
0,María,López,4000.00
1,Sofía,Torres,2100.75
2,Ana,García,1500.00
3,Luis,Martínez,800.50
4,Laura,Díaz,750.00
5,Pedro,Sánchez,500.00


## Reflexión Final

**Sistema elegido:** Gestión de Clientes

**¿Cómo funciona dentro del sistema web?**
La tabla `clientes` almacena la información de cada persona registrada.
Cada vez que alguien se registra, se ejecuta un INSERT.
Cuando consultan su perfil, se ejecuta un SELECT.

**¿Qué información almacena?**
Nombre, apellido, email, teléfono, saldo disponible y estado de la cuenta.

**¿Qué pantallas necesitaría el sistema?**
- 📋 Lista de clientes (SELECT)
- ➕ Formulario de registro (INSERT)
- ✏️ Formulario de edición (UPDATE)
- 🗑️ Confirmación de baja (DELETE)
- 🔍 Buscador por nombre o email (SELECT + WHERE)